<a href="https://colab.research.google.com/github/yeeshukanda-source/AAI2025/blob/main/Agentic_AI_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
!pip install -q langchain langgraph langchain-google-genai

In [37]:
from google.colab import userdata

google_key = userdata.get("GOOGLE_API_KEY").strip()

print("Google API secret found:", bool(google_key))

Google API secret found: True


In [38]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=google_key,
    temperature=0
)

response = llm.invoke("Say exactly: GEMINI IS WORKING")

print(response.content)

GEMINI IS WORKING


In [39]:
from typing import TypedDict, Annotated
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver


# ==========================================
# SYSTEM PROMPT
# ==========================================

SYSTEM_PROMPT = """
You are a helpful customer service assistant for an online electronics store.

Your job is to help customers with:
1. Order status and delivery questions
2. Refund and return questions
3. Product recommendations

Be polite, clear, helpful, and concise.
Use information from earlier messages when appropriate.
"""


# ==========================================
# CONVERSATION STATE
# ==========================================

class ChatState(TypedDict):
    messages: Annotated[list, add_messages]


# ==========================================
# AGENT 1: ORDER STATUS
# ==========================================

def order_agent(state: ChatState):

    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT + """
You are the Order Status Agent.
Help customers with orders, shipping, tracking,
and estimated delivery.

This is a classroom prototype.
Do not claim access to a real order database.
"""),
        *state["messages"]
    ])

    print("AGENT USED: ORDER STATUS AGENT")

    return {"messages": [response]}


# ==========================================
# AGENT 2: REFUND POLICY
# ==========================================

def refund_agent(state: ChatState):

    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT + """
You are the Refund Policy Agent.

Store policy:
- Returns are accepted within 30 days of delivery.
- Products should be in their original condition.
- Refunds are processed after the returned item is received.

Help customers understand returns and refunds.
"""),
        *state["messages"]
    ])

    print("AGENT USED: REFUND POLICY AGENT")

    return {"messages": [response]}


# ==========================================
# AGENT 3: PRODUCT RECOMMENDATION
# ==========================================

def product_agent(state: ChatState):

    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT + """
You are the Product Recommendation Agent.

Help customers choose electronics based on their
needs, preferences, and budget.

Give simple recommendations and briefly explain
why each recommendation is suitable.
"""),
        *state["messages"]
    ])

    print("AGENT USED: PRODUCT RECOMMENDATION AGENT")

    return {"messages": [response]}


# ==========================================
# ROUTER
# ==========================================

def router(state: ChatState):

    message = state["messages"][-1].content.lower()

    if any(word in message for word in
           ["order", "tracking", "delivery", "arrive", "shipping", "shipped"]):

        return "order"

    elif any(word in message for word in
             ["refund", "return", "money back"]):

        return "refund"

    else:

        return "product"


# ==========================================
# BUILD LANGGRAPH
# ==========================================

builder = StateGraph(ChatState)

builder.add_node("order_agent", order_agent)
builder.add_node("refund_agent", refund_agent)
builder.add_node("product_agent", product_agent)

builder.add_conditional_edges(
    START,
    router,
    {
        "order": "order_agent",
        "refund": "refund_agent",
        "product": "product_agent"
    }
)

builder.add_edge("order_agent", END)
builder.add_edge("refund_agent", END)
builder.add_edge("product_agent", END)


# ==========================================
# MEMORYSAVER
# ==========================================

memory = MemorySaver()

chatbot = builder.compile(checkpointer=memory)

config = {
    "configurable": {
        "thread_id": "customer-001"
    }
}


# ==========================================
# CHAT FUNCTION
# ==========================================

def chat(message):

    print("\nCUSTOMER:", message)

    result = chatbot.invoke(
        {"messages": [HumanMessage(content=message)]},
        config=config
    )

    print("\nCHATBOT:")
    print(result["messages"][-1].content)


print("SUCCESS: Agentic AI Customer Service Chatbot created!")

SUCCESS: Agentic AI Customer Service Chatbot created!


In [40]:
print("========== TEST 1: ORDER STATUS ==========")
chat("Where is my order #12345?")

print("\n========== TEST 2: REFUND POLICY ==========")
chat("What is your refund policy?")

print("\n========== TEST 3: PRODUCT RECOMMENDATION ==========")
chat("I need headphones under $100 for listening to music. What do you recommend?")

========== TEST 1: ORDER STATUS ==========

CUSTOMER: Where is my order #12345?
AGENT USED: ORDER STATUS AGENT

CHATBOT:
Thank you for reaching out about your order #12345!

As a prototype, I don't have access to real-time order tracking information. However, if I were a live system, I would typically tell you:

*   **Your order's current status** (e.g., "Processing," "Shipped," "Out for Delivery")
*   **Its estimated delivery date**
*   **A tracking number and link** to the carrier's website

Please let me know if you have any other questions about order processes or general delivery information!

========== TEST 2: REFUND POLICY ==========

CUSTOMER: What is your refund policy?
AGENT USED: REFUND POLICY AGENT

CHATBOT:
Our refund policy is as follows:

*   **Returns are accepted within 30 days of delivery.**
*   **Products must be in their original condition** to be eligible for a return.
*   **Refunds are processed after the returned item is received** and inspected by our team.

If

In [41]:
print("========== MEMORY TEST ==========")

chat("My budget was under $100. Which headphone option would be best for music?")

========== MEMORY TEST ==========

CUSTOMER: My budget was under $100. Which headphone option would be best for music?
AGENT USED: PRODUCT RECOMMENDATION AGENT

CHATBOT:
Considering your budget is under $100 and your primary use is listening to music, I would lean towards the **Sony WH-CH520 Wireless Headphones**.

*   **Why:** Over-ear headphones like the Sony WH-CH520 generally offer a more immersive sound experience and better comfort for extended music listening sessions compared to earbuds. They often provide a fuller soundstage, which many find preferable for enjoying music.


In [42]:
chat("Can I return it if I don't like it?")


CUSTOMER: Can I return it if I don't like it?
AGENT USED: REFUND POLICY AGENT

CHATBOT:
Yes, you can return it if you don't like it, provided it meets our return policy criteria:

*   **You must initiate the return within 30 days of delivery.**
*   **The headphones should be in their original condition.**

Once we receive the returned item and confirm its condition, your refund will be processed.
